In [79]:
# # Multi-Radius Monte Carlo Circular Scan-Area Sampling

# =============================================================
# 0. Initialize
# =============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
import matplotlib.pyplot as plt
import os
from joblib import Parallel, delayed
import multiprocessing

# --- Configuration ---
input_folder = "data_multi_radius"
boundary_file = os.path.join(input_folder, "boundary.shp")
lineaments_file = os.path.join(input_folder, "lineaments.shp")
output_folder = "results_multi_radius"
os.makedirs(output_folder, exist_ok=True)

# Base parameters
n_iterations = 100          # Monte Carlo iterations per radius
n_points_target = 100       # Circles per iteration
radius_min = 7000              # Minimum circle radius
radius_max = 70000             # Maximum circle radius
n_steps = 10                 # Number of radii
spacing_type = "linear"  # Options: "linear", "exponential", "log"

# Compute radius list
if spacing_type == "linear":
    radius_list = np.linspace(radius_min, radius_max, n_steps)
elif spacing_type == "exponential":
    radius_list = np.geomspace(radius_min, radius_max, n_steps)
elif spacing_type == "log":
    radius_list = np.logspace(np.log10(radius_min), np.log10(radius_max), n_steps)
else:
    raise ValueError("Invalid spacing_type: choose 'linear', 'exponential', or 'log'")

print(f"Radii to process ({spacing_type} spacing): {np.round(radius_list, 2)}")

# Parallelization setup
n_cores = max(multiprocessing.cpu_count() - 1, 1)
print(f"Using {n_cores} CPU cores for parallel execution.")

Radii to process (linear spacing): [ 7000. 14000. 21000. 28000. 35000. 42000. 49000. 56000. 63000. 70000.]
Using 19 CPU cores for parallel execution.


In [80]:
# =============================================================
# 1. Load Data
# =============================================================

poly_gdf = gpd.read_file(boundary_file)
lineaments_gdf = gpd.read_file(lineaments_file).to_crs(poly_gdf.crs)

print(f"✅ Loaded data: {len(poly_gdf)} polygons, {len(lineaments_gdf)} lineaments")

✅ Loaded data: 1 polygons, 43 lineaments


In [81]:
# =============================================================
# 2. Define Helper Function
# =============================================================

def run_iteration(iter_id, inner_poly, outer_poly, lineaments_gdf, n_points_target, circle_radius, crs):
    from shapely.geometry import Point

    # --- Random points inside inner polygon ---
    minx, miny, maxx, maxy = inner_poly.bounds
    bbox_area = (maxx - minx) * (maxy - miny)
    poly_area = inner_poly.area
    scaling_factor = poly_area / bbox_area * 1.1 if bbox_area > 0 else np.nan

    # Handle empty or invalid inner polygon
    if poly_area == 0 or np.isnan(scaling_factor) or np.isinf(scaling_factor):
        # Return circles_gdf with zero clipped_length and other columns
        circles_gdf = gpd.GeoDataFrame(
            geometry=[], crs=crs
        )
        circles_gdf["circle_id"] = []
        circles_gdf["iteration_id"] = []
        circles_gdf["clipped_length"] = []
        circles_gdf["radius"] = []
        circles_gdf["area"] = []
        circles_gdf["length_per_area"] = []
        circles_gdf["x"] = []
        circles_gdf["y"] = []
        circles_gdf["inside_outer"] = []
        return circles_gdf

    n_points_to_generate = int(n_points_target / scaling_factor * 1.5) if not np.isnan(scaling_factor) and not np.isinf(scaling_factor) else n_points_target * 2

    xs = np.random.uniform(minx, maxx, n_points_to_generate)
    ys = np.random.uniform(miny, maxy, n_points_to_generate)
    points = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(xs, ys)], crs=crs)
    points_within = points[points.within(inner_poly)].reset_index(drop=True)
    points_sample = points_within.sample(n_points_target, random_state=None).reset_index(drop=True)

    # --- Circles ---
    circles_gdf = gpd.GeoDataFrame(
        geometry=points_sample.geometry.buffer(circle_radius),
        crs=crs
    ).reset_index(drop=True)
    circles_gdf["circle_id"] = circles_gdf.index
    circles_gdf["iteration_id"] = iter_id

    # --- Spatial join with lineaments ---
    joined = gpd.sjoin(lineaments_gdf, circles_gdf[["circle_id", "geometry"]], predicate="intersects")

    # If no lineaments intersect any circles, return circles_gdf with zero clipped_length
    if joined.empty:
        circles_gdf["clipped_length"] = 0
        circles_gdf["radius"] = circle_radius
        circles_gdf["area"] = circles_gdf.geometry.area
        circles_gdf["length_per_area"] = 0
        circles_gdf["x"] = points_sample.geometry.x
        circles_gdf["y"] = points_sample.geometry.y
        circles_gdf["inside_outer"] = circles_gdf.geometry.within(outer_poly)
        return circles_gdf

    # --- Intersection and length ---
    from shapely.geometry import MultiLineString, GeometryCollection, LineString, base
    def safe_intersection(row):
        try:
            geom = row.geometry.intersection(circles_gdf.loc[row.circle_id, "geometry"])
            if geom.is_empty:
                return None
            if isinstance(geom, GeometryCollection):
                lines = [g for g in geom.geoms if isinstance(g, LineString)]
                if len(lines) == 0:
                    return None
                if len(lines) == 1:
                    return lines[0]
                return MultiLineString(lines)
            return geom
        except Exception:
            return None
    joined["clipped_geom"] = joined.apply(safe_intersection, axis=1)
    # Filter to only valid Shapely geometry objects
    joined = joined[joined["clipped_geom"].apply(lambda x: isinstance(x, base.BaseGeometry))]
    joined["clipped_length"] = joined["clipped_geom"].apply(lambda x: x.length if x is not None else 0)

    # --- Aggregate and merge ---
    lengths_per_circle = joined.groupby("circle_id")["clipped_length"].sum().reset_index()
    circles_gdf = circles_gdf.merge(lengths_per_circle, on="circle_id", how="left")
    circles_gdf["clipped_length"] = circles_gdf["clipped_length"].fillna(0)
    circles_gdf["radius"] = circle_radius
    circles_gdf["area"] = circles_gdf.geometry.area
    circles_gdf["length_per_area"] = circles_gdf["clipped_length"] / circles_gdf["area"]
    circles_gdf["x"] = points_sample.geometry.x
    circles_gdf["y"] = points_sample.geometry.y
    circles_gdf["inside_outer"] = circles_gdf.geometry.within(outer_poly)

    return circles_gdf

In [82]:
# =============================================================
# 3. Parallel Radius Loop with Figure Saving
# =============================================================

def process_radius(radius):
    print(f"\n▶️ Radius = {radius:.2f}")
    buffer_distance = -radius

    # Inner and outer polygons
    poly_gdf["inner_polygon"] = poly_gdf.geometry.buffer(buffer_distance)
    inner_poly = poly_gdf["inner_polygon"].unary_union
    outer_poly = poly_gdf.geometry.unary_union

    # Run all iterations (sequential within this radius)
    results_list = []
    for i in range(1, n_iterations + 1):
        result = run_iteration(i, inner_poly, outer_poly, lineaments_gdf, n_points_target, radius, poly_gdf.crs)
        result["radius_tested"] = radius
        results_list.append(result)
        last_iteration_result = result

    # Combine iterations for this radius
    results_gdf = pd.concat(results_list, ignore_index=True)

    # Save CSV
    csv_path = os.path.join(output_folder, f"circle_density_r{radius:.2f}.csv")
    results_gdf.drop(columns="geometry").to_csv(csv_path, index=False)

    # --- Plot colored by length_per_area ---
    fig, ax = plt.subplots(figsize=(8, 8))
    poly_gdf.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1.2, label="Outer Polygon")
    gpd.GeoSeries(inner_poly).plot(ax=ax, facecolor='none', edgecolor='red', linestyle='--', label="Inner Polygon")
    lineaments_gdf.plot(ax=ax, color='gray', linewidth=0.5, label="Lineaments")
    last_iteration_result.plot(
        ax=ax,
        column="length_per_area",
        cmap="viridis",
        legend=True,
        alpha=0.6,
        edgecolor='black',
        linewidth=0.3
    )
    plt.title(f"Radius = {radius:.2f} m — Last Iteration (Colored by Density)")
    plt.legend()
    plt.axis("equal")
    plt.tight_layout()

    # Save figures
    fig_path_png = os.path.join(output_folder, f"figure_r{radius:.2f}.png")
    fig_path_svg = os.path.join(output_folder, f"figure_r{radius:.2f}.svg")
    plt.savefig(fig_path_png, dpi=300)
    plt.savefig(fig_path_svg, format='svg')
    plt.close()

    print(f"✅ Saved CSV and figures for radius {radius:.2f}")
    return results_gdf


# --- Run in parallel for all radii ---
all_results = Parallel(n_jobs=n_cores, verbose=5)(
    delayed(process_radius)(r) for r in radius_list
)

# Combine all results
master_df = pd.concat(all_results, ignore_index=True)

# Save combined CSV
master_csv = os.path.join(output_folder, f"circle_density_ALL_{spacing_type}_spacing.csv")
master_df.drop(columns="geometry").to_csv(master_csv, index=False)

print(f"\n🏁 All radii processed. Combined CSV saved to:\n{master_csv}")

[Parallel(n_jobs=19)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=19)]: Done   3 out of  10 | elapsed:    6.8s remaining:   16.1s
[Parallel(n_jobs=19)]: Done   6 out of  10 | elapsed:   16.3s remaining:   10.9s
[Parallel(n_jobs=19)]: Done  10 out of  10 | elapsed:   30.0s finished



🏁 All radii processed. Combined CSV saved to:
results_multi_radius\circle_density_ALL_linear_spacing.csv


In [83]:
# =============================================================
# 4. Histogram of length_per_area for each radius
# =============================================================

import seaborn as sns

for radius in sorted(master_df['radius_tested'].unique()):
    subset = master_df[master_df['radius_tested'] == radius]
    plt.figure(figsize=(8, 6))
    sns.histplot(subset['length_per_area'], bins=30, kde=True, color='royalblue')
    plt.title(f'Histogram of length_per_area for radius {radius:.2f}')
    plt.xlabel('length_per_area')
    plt.ylabel('Count')
    plt.tight_layout()
    hist_png = os.path.join(output_folder, f'histogram_length_per_area_r{radius:.2f}.png')
    hist_svg = os.path.join(output_folder, f'histogram_length_per_area_r{radius:.2f}.svg')
    plt.savefig(hist_png, dpi=300)
    plt.savefig(hist_svg, format='svg')
    plt.close()
print('✅ Histograms of length_per_area saved for all radii.')

✅ Histograms of length_per_area saved for all radii.


In [84]:
# =============================================================
# 5. Box-and-whisker plot of length_per_area for all radii
# =============================================================

plt.figure(figsize=(12, 7))
sns.boxplot(x='radius_tested', y='length_per_area', data=master_df, color='lightblue')
plt.title('Box-and-Whisker Plot of length_per_area by Circle Radius')
plt.xlabel('Circle Radius')
plt.ylabel('length_per_area')
plt.xticks(rotation=45)
plt.tight_layout()
boxplot_png = os.path.join(output_folder, 'boxplot_length_per_area_by_radius.png')
boxplot_svg = os.path.join(output_folder, 'boxplot_length_per_area_by_radius.svg')
plt.savefig(boxplot_png, dpi=300)
plt.savefig(boxplot_svg, format='svg')
plt.close()
print('✅ Box-and-whisker plot of length_per_area by radius saved as PNG and SVG.')


✅ Box-and-whisker plot of length_per_area by radius saved as PNG and SVG.
